# TARDIS
This notebook loads the dataset, cleans the data, handles missing values, and creates new features for further analysis.

In [22]:
import pandas as pd
import numpy as np

## Load and Inspect the Dataset

In [23]:
df = pd.read_csv("dataset.csv", sep=';')
print("(Nb rows, Nb columns): ", df.shape)
df.info()

(Nb rows, Nb columns):  (12070, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                         Non-Null Count  Dtype 
---  ------                                                                         --------------  ----- 
 0   Date                                                                           12010 non-null  object
 1   Service                                                                        11830 non-null  object
 2   Departure station                                                              12011 non-null  object
 3   Arrival station                                                                12011 non-null  object
 4   Average journey time                                                           11830 non-null  object
 5   Number of scheduled trains                                                     11830 non-null  object

In [24]:
df.head()

,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Cancellation comments,Number of trains delayed at departure,Average delay of late trains at departure,...,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,...,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,...,32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,...,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,...,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,...,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


In [25]:
print(df.columns)

Index(['Date', 'Service', 'Departure station', 'Arrival station',
       'Average journey time', 'Number of scheduled trains',
       'Number of cancelled trains', 'Cancellation comments',
       'Number of trains delayed at departure',
       'Average delay of late trains at departure',
       'Average delay of all trains at departure', 'Departure delay comments',
       'Number of trains delayed at arrival',
       'Average delay of late trains at arrival',
       'Average delay of all trains at arrival', 'Arrival delay comments',
       'Number of trains delayed > 15min',
       'Average delay of trains > 15min (if competing with flights)',
       'Number of trains delayed > 30min', 'Number of trains delayed > 60min',
       'Pct delay due to external causes', 'Pct delay due to infrastructure',
       'Pct delay due to traffic management', 'Pct delay due to rolling stock',
       'Pct delay due to station management and equipment reuse',
       'Pct delay due to passenger handling (

In [26]:
nb_duplicates = df.duplicated().sum()

if nb_duplicates > 0:
    df = df.drop_duplicates()

print("Nb duplicates: ", nb_duplicates)
print("(Nb rows, Nb columns): ", df.shape)

Nb duplicates:  174
(Nb rows, Nb columns):  (11896, 26)


## Data Cleaning

In [27]:
df['Date'] = df['Date'].str.replace('/', '-')
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m', errors="coerce")

numeric_cols = ['Average journey time',
                'Number of scheduled trains',
                'Number of cancelled trains',
                'Number of trains delayed at departure',
                'Average delay of late trains at departure',
                'Average delay of all trains at departure',
                'Number of trains delayed at arrival',
                'Average delay of late trains at arrival',
                'Average delay of all trains at arrival',
                'Number of trains delayed > 15min',
                'Average delay of trains > 15min (if competing with flights)',
                'Number of trains delayed > 30min',
                'Number of trains delayed > 60min',
                'Pct delay due to external causes',
                'Pct delay due to infrastructure',
                'Pct delay due to traffic management',
                'Pct delay due to rolling stock',
                'Pct delay due to station management and equipment reuse',
                'Pct delay due to passenger handling (crowding, disabled persons, connections)']

for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(0)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11896 entries, 0 to 12060
Data columns (total 26 columns):
 #   Column                                                                         Non-Null Count  Dtype         
---  ------                                                                         --------------  -----         
 0   Date                                                                           11594 non-null  datetime64[ns]
 1   Service                                                                        11656 non-null  object        
 2   Departure station                                                              11837 non-null  object        
 3   Arrival station                                                                11837 non-null  object        
 4   Average journey time                                                           11896 non-null  float64       
 5   Number of scheduled trains                                                     11896 n

In [28]:
print(df.isnull().sum())

Date                                                                               302
Service                                                                            240
Departure station                                                                   59
Arrival station                                                                     59
Average journey time                                                                 0
Number of scheduled trains                                                           0
Number of cancelled trains                                                           0
Cancellation comments                                                            11324
Number of trains delayed at departure                                                0
Average delay of late trains at departure                                            0
Average delay of all trains at departure                                             0
Departure delay comments                   

The dataset is checked for missing values so that incomplete rows or invalid values can be handled before feature creation.

In [29]:
df = df.dropna(subset=["date", "scheduled_time", "actual_time", "delay_minutes"])
print("(Nb rows, Nb columns): ", df.shape)

KeyError: ['date', 'scheduled_time', 'actual_time', 'delay_minutes']

Rows with missing values in the key columns are removed because these fields are necessary for reliable analysis and later feature engineering.

## Feature Engineering

New features are created from the original columns to make the dataset easier to analyze. In this section, temporal features such as year, month, quarter, season, and scheduled hour are added, as well as route, peak-hour, and delay-category features.

In [ ]:
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month

In [ ]:
df["Month name"] = df["date"].dt.month_name()
df["Quarter"] = df["date"].dt.quarter

In [ ]:
df["Season"] = df["Month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

In [ ]:
df["Route"] = df["departure_station"] + " -> " + df["arrival_station"]

In [ ]:
df["scheduled_hour"] = df["scheduled_time"].apply(
    lambda x: x.hour if pd.notnull(x) else np.nan
)

In [ ]:
df["is_peak_hour"] = df["scheduled_hour"].apply(
    lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0
)

In [ ]:
df["delay_category"] = pd.cut(
    df["delay_minutes"],
    bins=[-1, 5, 15, 30, float("inf")],
    labels=["On time / slight", "Short delay", "Medium delay", "Long delay"]
)

In [ ]:
df[["Year", "Month", "scheduled_hour", "is_peak_hour", "delay_category", "Route"]].head()

In [ ]:
df.to_csv('cleaned_dataset.csv', index=False)

## Delay Minutes Analysis

In [ ]:
df["delay_minutes"].describe()

In [ ]:
print("Mean delay:", df["delay_minutes"].mean())
print("Median delay:", df["delay_minutes"].median())
print("Minimum delay:", df["delay_minutes"].min())
print("Maximum delay:", df["delay_minutes"].max())

In [ ]:
df["delay_minutes"].plot(kind="hist", bins=10, title="Distribution of delay minutes")

The average delay is 18 minutes and the median delay is 15 minutes. This suggests that most delays are moderate, but some larger delays may increase the average. The delay values range from 5 to 35 minutes, showing that the dataset includes both shorter and longer delays.

In [ ]:
df["delay_minutes"].plot(kind="box", title="Boxplot of delay minutes")

In [ ]:
df.sort_values("delay_minutes", ascending=False).head()

## Average Delay by Departure Station

In [ ]:
departure_delay = df.groupby("departure_station")["delay_minutes"].mean().sort_values(ascending=False)
departure_delay

In [ ]:
departure_delay.plot(kind="bar", title="Average Delay by Departure Station")

This analysis compares the average delay across departure stations. Grouping the data by `departure_station` helps identify whether some stations tend to have higher delays than others. The bar chart makes it easier to compare stations and observe which ones have the highest average delay.